# 03 - Distributed Analytics and SQL Serving Prep (Lab Support)

Use this notebook to prototype Spark-style aggregates locally and prepare Athena-facing query logic.

In [ ]:
from pathlib import Path
import pandas as pd

if Path.cwd().name == 'notebooks':
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

raw_files = sorted((REPO_ROOT / 'data/raw').glob('trips_2023_*.csv'))
zones_file = REPO_ROOT / 'data' / 'reference' / 'zone_lookup.csv'

print('Raw partitions:', [f.name for f in raw_files])
print('Zone lookup file exists:', zones_file.exists())

In [ ]:
def clean_minimal(df: pd.DataFrame) -> pd.DataFrame:
    x = df.rename(columns={
        'pickup_ts': 'pickup_datetime',
        'pickup_zone': 'pickup_zone_id',
        'dropoff_zone': 'dropoff_zone_id',
        'duration_min': 'trip_duration_min',
    }).copy()

    keep = ['pickup_datetime', 'pickup_zone_id', 'dropoff_zone_id', 'trip_distance', 'trip_duration_min', 'fare_amount']
    x = x[keep]

    x['pickup_datetime'] = pd.to_datetime(x['pickup_datetime'], errors='coerce')
    for col in ['pickup_zone_id', 'dropoff_zone_id', 'trip_distance', 'trip_duration_min', 'fare_amount']:
        x[col] = pd.to_numeric(x[col], errors='coerce')

    x = x.dropna().copy()
    x = x[(x['trip_duration_min'].between(1, 180)) & (x['trip_distance'].between(0.1, 100)) & (x['fare_amount'].between(0, 500))]
    x['pickup_zone_id'] = x['pickup_zone_id'].astype(int)
    return x

In [ ]:
raw_df = pd.concat([pd.read_csv(f) for f in raw_files], ignore_index=True)
clean_df = clean_minimal(raw_df)
zones = pd.read_csv(zones_file)

enriched = clean_df.merge(
    zones.rename(columns={'zone_id': 'pickup_zone_id', 'zone_name': 'pickup_zone'}),
    on='pickup_zone_id',
    how='left'
)
enriched['pickup_zone'] = enriched['pickup_zone'].fillna('Unknown Zone')
enriched['date'] = enriched['pickup_datetime'].dt.strftime('%Y-%m-%d')
enriched['hour_of_day'] = enriched['pickup_datetime'].dt.hour

enriched.head()

In [ ]:
hourly = (
    enriched.groupby(['pickup_zone', 'hour_of_day'], as_index=False)
    .agg(
        trip_count=('fare_amount', 'count'),
        avg_fare=('fare_amount', 'mean'),
        avg_duration_min=('trip_duration_min', 'mean'),
        p90_fare=('fare_amount', lambda s: s.quantile(0.9)),
    )
    .sort_values(['pickup_zone', 'hour_of_day'])
)

for col in ['avg_fare', 'avg_duration_min', 'p90_fare']:
    hourly[col] = hourly[col].round(2)

hourly.head(12)

In [ ]:
daily = (
    enriched.groupby(['pickup_zone', 'date'], as_index=False)
    .agg(total_revenue=('fare_amount', 'sum'), trip_volume=('fare_amount', 'count'))
    .sort_values(['date', 'pickup_zone'])
)
daily['total_revenue'] = daily['total_revenue'].round(2)
daily.head(12)

In [ ]:
print('Top zones by volume')
(
    hourly.groupby('pickup_zone', as_index=False)['trip_count']
    .sum()
    .sort_values('trip_count', ascending=False)
    .head(10)
)

In [ ]:
print('Average fare by hour of day')
(
    hourly.groupby('hour_of_day', as_index=False)['avg_fare']
    .mean()
    .assign(avg_fare=lambda d: d['avg_fare'].round(2))
    .sort_values('hour_of_day')
)

In [ ]:
example_date = daily['date'].iloc[0] if len(daily) else '2023-01-15'
all_rows = len(daily)
filtered_rows = int((daily['date'] == example_date).sum())

print('Partition-pruning intuition (row-level proxy)')
print('Rows scanned without filter:', all_rows)
print(f'Rows scanned with date={example_date}:', filtered_rows)

In [ ]:
athena_sql = '''
CREATE DATABASE IF NOT EXISTS lab2_analytics;

CREATE EXTERNAL TABLE IF NOT EXISTS lab2_analytics.hourly_trips (
  pickup_zone STRING,
  hour_of_day INT,
  trip_count BIGINT,
  avg_fare DOUBLE,
  avg_duration_min DOUBLE,
  p90_fare DOUBLE
)
STORED AS PARQUET
LOCATION 's3://<BUCKET>/aggregated/hourly/';

CREATE EXTERNAL TABLE IF NOT EXISTS lab2_analytics.daily_summary (
  pickup_zone STRING,
  total_revenue DOUBLE,
  trip_volume BIGINT
)
PARTITIONED BY (date STRING)
STORED AS PARQUET
LOCATION 's3://<BUCKET>/aggregated/daily_summary/';

MSCK REPAIR TABLE lab2_analytics.daily_summary;
'''.strip()

print(athena_sql)

## Reflection Prompt
1. Which aggregations are cheap locally but expensive at real scale?
2. Why does partitioning improve query efficiency in Athena?
3. Which outputs should be treated as analyst-facing contracts?